[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Relaciones entre variables.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo II: *Describir y comparar*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se medirán relaciones entre variables con las correlaciones de Pearson y Spearman, sin confundir correlación con causalidad, y se estudiarán los confusores y la paradoja de Simpson.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien termina antes o quiere profundizar. Si no da tiempo en clase, se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencia de la API: curso py311.

La API que se usa hoy está explicada método por método en py311; aquí solo se repasa y el énfasis está en **interpretar** y en **decidir** qué tabla, qué medida de asociación y qué gráfica responden mejor la pregunta:

| Tema | *Notebooks* de py311 |
|---|---|
| `melt`, `stack`/`unstack`, `pivot_table`, `crosstab` | [21 Transformación de *dataframes* y series](https://github.com/PythonistaMX/py311/blob/main/21_transformacion_de_objetos.ipynb) |
| `groupby`, agregación y `transform` por grupo | [24 Métodos `groupby()`](https://github.com/PythonistaMX/py311/blob/main/24_metodos_groupby.ipynb) |
| *Seaborn*: funciones por tipo de gráfica y la interfaz de objetos | [31 Introducción a *Seaborn*](https://github.com/PythonistaMX/py311/blob/main/31_introduccion_a_seaborn.ipynb) · [32 Objetos de *Seaborn*](https://github.com/PythonistaMX/py311/blob/main/32_objetos_de_seaborn.ipynb) |

> ⚠️ **Nota:** py311 se escribió para *pandas* 2 y este curso usa *pandas* 3, así que pueden verse diferencias (por ejemplo, texto como `object` en lugar de `str`, o el uso de `inplace=True`). Cuando difieran, se sigue lo que muestra este *notebook* (ver la sección 1.1 de `03_pandas_3_esencial.ipynb`).

### Preparación del entorno.

* La siguiente celda importará las bibliotecas, incluido el módulo `stats` de *SciPy*, y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats

warnings.filterwarnings("ignore", message="vert: bool was deprecated")

**Nota:** el filtro de advertencias oculta solo un aviso interno de *Seaborn* 0.13 con *Matplotlib* 3.11 al dibujar diagramas de caja (`vert: bool was deprecated`). No afecta los resultados.

* La siguiente celda leerá `penguins_limpio.csv` o lo regenerará desde `penguins.csv` si no existe.

In [ ]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio).astype({"species": "category", "island": "category", "sex": "category"})
pinguinos.shape

`tips` registra 244 cuentas de un restaurante: `total_bill` (cuenta, en dólares), `tip` (propina), `sex` y `smoker` de quien paga, `day`, `time` (comida o cena) y `size` (personas en la mesa). Se creará la **propina porcentual** `tip_pct = 100 · tip / total_bill` y se ordenarán los días.

* La siguiente celda leerá `tips.csv`, creará `tip_pct` y convertirá `day` y `time` en categorías ordenadas.

In [ ]:
propinas = (
    pd.read_csv(os.path.join(DATA, "tips.csv"))
    .assign(tip_pct=lambda d: 100 * d["tip"] / d["total_bill"])
    .astype({"day": pd.CategoricalDtype(["Thur", "Fri", "Sat", "Sun"], ordered=True),
             "time": pd.CategoricalDtype(["Lunch", "Dinner"], ordered=True)})
)
propinas.head()

---
## 1. Relaciones entre variables.

| Medida | Mide | Rango | Sensible a |
|---|---|---|---|
| **Pearson** (*r*) | Asociación **lineal** | −1 a 1 | Atípicos; relaciones no lineales |
| **Spearman** (ρ) | Asociación **monótona** (Pearson sobre las posiciones) | −1 a 1 | Menos sensible a atípicos |

Ambas se calculan con `df.corr(method="pearson")` o `method="spearman"`, o con `stats.pearsonr` y `stats.spearmanr`. Para dos variables **categóricas** se usa una tabla de contingencia (y en `21_proporciones_y_mas_de_dos_grupos`, la prueba χ²).

**Concepto: qué mide una correlación, en palabras.** Pearson responde **qué tan bien se acomodan los puntos sobre una recta**: 1 es una recta perfecta que sube, −1 una que baja y 0 indica que no hay tendencia en línea recta. Por dentro, revisa cada observación: si está del mismo lado del promedio en las dos variables (las dos arriba o las dos abajo), empuja la correlación hacia arriba; si está arriba en una y abajo en la otra, la empuja hacia abajo.

Spearman primero sustituye cada valor por su **posición** en la lista ordenada (1.º, 2.º, 3.º…, lo que se llama *rango*) y luego calcula la correlación con esas posiciones. Como solo usa el orden, un valor enorme cuenta igual que uno apenas mayor, y por eso casi no le afecta un atípico. Mide si, cuando una variable sube, la otra tiende siempre a subir (o a bajar), aunque no sea en línea recta (relación **monótona**).

**Ejemplo:** en un grupo, las horas de estudio y el puntaje en un cuestionario de práctica suelen tener correlación positiva: quien estudia más que el promedio suele sacar más puntos que el promedio. Ninguno de los dos coeficientes mide la **pendiente**: una correlación de 0.9 puede ir con un aumento grande o diminuto, porque solo dice qué tan alineados están los puntos, no cuánto sube una variable por cada unidad de la otra.

* La siguiente celda calculará Pearson y Spearman entre la cuenta y la propina.

In [ ]:
{"pearson": round(float(propinas["total_bill"].corr(propinas["tip"])), 3),
 "spearman": round(float(propinas["total_bill"].corr(propinas["tip"], method="spearman")), 3)}

**Interpretación:** Pearson (0.676) y Spearman (0.679) casi coinciden: la relación es aproximadamente lineal y no la dominan unos pocos puntos. Una correlación de 0.68 dice que las cuentas grandes tienden a venir con propinas grandes, pero deja mucha variación: cuentas iguales reciben propinas muy distintas.

**¿Qué tan fuerte es una correlación?** No existe una regla universal. Como guía burda: de 0.1 a 0.3 es débil, de 0.3 a 0.5 moderada, de 0.5 a 0.7 entre moderada y fuerte, y más de 0.7 fuerte; cada campo tiene su costumbre. Con esa guía, 0.68 queda entre moderada y fuerte.

* La siguiente celda dibujará la relación con su recta de regresión.

In [ ]:
ax = sns.regplot(propinas, x="total_bill", y="tip", scatter_kws={"alpha": 0.5})
ax.set(xlabel="cuenta (USD)", ylabel="propina (USD)", title="A mayor cuenta, mayor propina (r = 0.68)")
plt.show()

**Ejemplo:** Pearson y Spearman difieren cuando la relación es monótona pero no lineal, o cuando hay atípicos. Se simularán ambos casos.

* La siguiente celda comparará ambas correlaciones en una relación exponencial y en datos sin relación con un solo atípico.

In [ ]:
x = np.arange(1, 21, dtype=float)
exponencial = pd.DataFrame({"x": x, "y": np.exp(x / 3)})
sin_relacion = pd.DataFrame({"x": rng.normal(size=30), "y": rng.normal(size=30)})
sin_relacion.loc[30] = [8.0, 8.0]
pd.DataFrame({
    "exponencial": [exponencial.corr().iloc[0, 1], exponencial.corr("spearman").iloc[0, 1]],
    "ruido + 1 atípico": [sin_relacion.corr().iloc[0, 1], sin_relacion.corr("spearman").iloc[0, 1]],
}, index=["pearson", "spearman"]).round(2)

En la relación exponencial, Spearman vale 1 (la relación es perfectamente monótona) y Pearson solo 0.8 (no es lineal). En los datos sin relación, **un solo punto** lleva a Pearson a 0.72, mientras que Spearman sigue cerca de 0. Conviene calcular ambas y, sobre todo, **graficar**.

> 💡 **Una correlación no es un porcentaje.** Una correlación de 0.9 no significa que «el 90% del resultado se explique» por la otra variable, ni que la pendiente sea 0.9. Solo dice qué tan alineados están los puntos. Para hablar de «qué parte de la variación acompaña» se eleva *r* al cuadrado: 0.9 deja 0.81, y el 19% restante sigue sin explicarse.

### 1.1 Correlación no es causalidad.

Una correlación puede deberse a:
* **Causalidad directa** (X causa Y).
* **Causalidad inversa** (Y causa X).
* Un **confusor**: una tercera variable Z causa ambas.
* **Selección** de la muestra o **azar**.

**Ejemplo:** se simulará un año de datos en el que la **temperatura** aumenta tanto la venta de helados como los ahogamientos. Helados y ahogamientos no se causan entre sí.

* La siguiente celda simulará 365 días de temperatura, venta de helados y ahogamientos.

In [ ]:
temperatura = rng.normal(22, 6, size=365)
dias = pd.DataFrame({
    "temperatura": temperatura,
    "helados": 50 + 8 * temperatura + rng.normal(0, 20, size=365),
    "ahogamientos": rng.poisson(np.clip(0.3 * temperatura - 3, 0.1, None)),
})
dias.corr().round(2)

**Interpretación:** helados y ahogamientos tienen una correlación de 0.64 aunque por construcción no se afectan entre sí. Ambos se correlacionan con la temperatura (0.92 y 0.72), que es la causa común.

> 💡 **Confusor:** es una tercera variable que influye en las otras dos y hace que parezcan relacionadas entre sí. Para descartarlo, se **compara a igual valor del confusor**: dentro de cada grupo, o con la misma temperatura.

* La siguiente celda quitará a helados y ahogamientos la parte que explica la temperatura (residuos de una recta ajustada con `np.polyfit`) y correlacionará lo que queda.

In [ ]:
def residuos(y, x):
    pendiente, ordenada = np.polyfit(x, y, deg=1)
    return y - (ordenada + pendiente * x)

round(float(np.corrcoef(residuos(dias["helados"], dias["temperatura"]),
                        residuos(dias["ahogamientos"], dias["temperatura"]))[0, 1]), 2)

Una vez descontado el efecto de la temperatura, la correlación entre helados y ahogamientos prácticamente desaparece: la temperatura era el **confusor**. Controlar por una variable significa comparar observaciones con el mismo valor de esa variable; los residuos son una forma de hacerlo y la paradoja de Simpson, a continuación, muestra otra (comparar **dentro** de cada grupo).

### 1.2 La paradoja de Simpson.

Una tendencia que aparece en todos los grupos **se invierte** (o desaparece) al juntar los grupos, porque los grupos tienen centros distintos. Es un caso extremo de confusión, y `penguins` tiene un ejemplo clásico que se resolverá en el ejercicio 2.

En la paradoja de Simpson, una relación que aparece en todos los grupos por separado cambia de signo (o desaparece) al juntar los grupos, porque los grupos están en lugares distintos de la gráfica. Cuando la pregunta es qué pasa **dentro** de cada grupo, la respuesta correcta es la de los grupos, no la global. Por eso, antes de interpretar una correlación global, conviene preguntarse si los datos mezclan grupos distintos.

### Ejercicio 1 · Matriz de correlaciones.

**🎯 Objetivo:** calcular y comparar Pearson y Spearman entre varias medidas. **Qué se espera aprender:** a resumir muchas relaciones a la vez y a detectar cuáles son casi la misma información.

Calcula las matrices de Pearson y de Spearman de las cuatro medidas de `pinguinos` y dibuja la de Pearson como mapa de calor (`sns.heatmap(..., annot=True, vmin=-1, vmax=1, cmap="vlag")`). ¿Qué par está más correlacionado? ¿Hay alguna correlación negativa?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### Ejercicio 2 · La paradoja de Simpson en los picos.

**🎯 Objetivo:** comparar una correlación global con las correlaciones por grupo. **Qué se espera aprender:** que mezclar grupos puede invertir el signo de una relación.

Calcula la correlación de Pearson entre `bill_length_mm` y `bill_depth_mm`:
1. Con todos los pingüinos juntos.
2. Dentro de cada especie (pista: `groupby("species")` y `apply` con una función que calcule la correlación).

¿Qué le pasa al signo? ¿Cuál de las dos respuestas describe mejor la relación entre largo y profundidad del pico?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### Reto 1 · ¿Las cuentas grandes dejan menos propina porcentual?

**🎯 Objetivo:** evaluar una relación con un valor extremo. **Qué se espera aprender:** a medir cuánto depende una correlación de un solo punto.

Calcula Pearson y Spearman entre `total_bill` y `tip_pct`, con y sin la cuenta de propina máxima (71%). Grafica la relación. ¿Qué medida es más estable? ¿Qué podría explicar la relación?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

📖 **Para profundizar (opcional).** *La fórmula de Pearson (sección 1).* Pearson es la covarianza estandarizada:

$$r = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_i (x_i - \bar{x})^2}\,\sqrt{\sum_i (y_i - \bar{y})^2}}$$

Cada producto del numerador es positivo cuando $x_i$ y $y_i$ están del mismo lado de sus medias. Spearman es la misma fórmula aplicada a los rangos.

**💡 Ideas clave de la sección 1:**
* Pearson mide asociación lineal; Spearman, monótona. Se calculan ambas y se grafica.
* Un solo atípico puede fabricar una correlación.
* Correlación no es causalidad: causalidad inversa, confusores, selección o azar. Controlar por una variable es comparar a igual valor de esa variable.

---
## ✅ Cierre.

**Práctica (en Classroom):** terminar los ejercicios pendientes de este *notebook* y, si es posible, un reto.

**Siguiente *notebook*:** `15_visualizacion_efectiva.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>